---
title: "📐The Hidden Geometry of Software Coupling (Part 1)"
date: 2026-03-09 00:11:50 -0500
categories:
  - sdlc
  - architecture
  - coupling
  - metrics
author: steven
---
<img alt="📐 The Hidden Geometry of Software Coupling" title="📐 The Hidden Geometry of Software Coupling"  src="../assets/images/metrics.png"/>





# 📐 The Hidden Geometry of Software Coupling
### Part 1 — The Metrics That Predict Architectural Failure


In [15]:
# ============================================================
# SETUP — Run this first!
# ============================================================

# Install plotly if needed (uncomment):
# !pip install  plotly pandas

import matplotlib as mpl
import matplotlib.pyplot as plt
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.graph_objs import Figure
from plotly.subplots import make_subplots
import numpy as np

print("✅ Plotly version:", go.__version__ if hasattr(go, '__version__') else 'installed')
print("✅ matplotlib version:", mpl.__version__ if hasattr(mpl, '__version__') else 'installed')
print("✅ Setup complete!")

# ============================================================
# 🎨 CONFIGURATION
# ============================================================

CONFIG = {
    # Background colors
    'blue_dot': '#396B8F',
    'paper_bg': '#FCF5E5',
    'plot_bg': 'rgba(252,245,229,0.8)',

    # Instability colorscale (list of [position, color])
    # Position: 0.0 to 1.0, Color: any CSS color
    'instability_colorscale': [
        [0.00, '#FFFFFF'],   # White at I=0
        [0.05, '#6AD8D8'],   # Light teal
        [0.20, '#0D3D3D'],   # Dark teal
        [0.30, '#E0B060'],   # Light bronze
        [0.45, '#5A3A12'],   # Dark bronze
        [0.50, '#3A0808'],   # Darkest red (center)
        [0.55, '#E89048'],   # Light copper
        [0.70, '#6A3210'],   # Dark copper
        [0.80, '#8A4A9A'],   # Light purple
        [0.95, '#2A0A3A'],   # Dark purple
        [1.00, '#000000'],   # Black at I=1
    ],

    # Distance colorscale (green=good, red=bad)
    'distance_colorscale': [
        [0.0, '#2E7D32'],    # Dark green (D=0, ideal)
        [0.2, '#4CAF50'],    # Green
        [0.4, '#8BC34A'],    # Light green
        [0.5, '#FFEB3B'],    # Yellow
        [0.7, '#FF9800'],    # Orange
        [0.85, '#F44336'],   # Red
        [1.0, '#B71C1C'],    # Dark red (D=1, worst)
    ],

    # Zone colors for Main Sequence
    'zone_pain': 'rgba(232,180,180,0.6)',
    'zone_useless': 'rgba(180,180,232,0.6)',
    'main_seq': '#4CAF50',

    # Default camera for 3D plots
    'camera': dict(
        eye=dict(x=1.5, y=1.5, z=1.2),
        up=dict(x=0, y=0, z=1)
    ),

    # Mesh resolution
    'mesh_points': 100,  # Lower than matplotlib for performance
}

PALETTE = {
    "paper_bg": "#EFE7D6",
    "plot_bg": "#E6DDCB",
    "grid": "rgba(70, 60, 45, 0.14)",
    "axis": "#2E4369",
    "zone_pain": "rgba(178, 94, 94, 0.22)",
    "zone_useless": "rgba(144, 134, 196, 0.24)",
    "main_seq": "#4DAA57",
    "guide": "rgba(90, 82, 70, 0.35)",
    "label_bg": "rgba(252,245,229,0.82)",
}

DETAILED_MODULES = [
    {"name": "Database Schema",        "I": 0.10, "A": 0.05, "color": "#E53935"},
    {"name": "Unused Interfaces",      "I": 0.70, "A": 0.95, "color": "#1E40FF"},
    {"name": "Core Domain Interfaces", "I": 0.22, "A": 0.78, "color": "#0B8A0B"},
    {"name": "Domain Model",           "I": 0.38, "A": 0.52, "color": "#1E8E1E"},
    {"name": "Service Layer",          "I": 0.68, "A": 0.36, "color": "#169C16"},
    {"name": "Perfect Balance",        "I": 0.50, "A": 0.50, "color": "#006400"},
    {"name": "API Gateway",            "I": 0.80, "A": 0.40, "color": "#F4A000"},
    {"name": "Shared Utilities",       "I": 0.34, "A": 0.44, "color": "#8A6D3B"},
    {"name": "Integration Adapter",    "I": 0.88, "A": 0.12, "color": "#8C564B"},
]



print("🎨 Configuration loaded")

✅ Plotly version: installed
✅ matplotlib version: 3.10.8
✅ Setup complete!
🎨 Configuration loaded


## 📃Introduction

Software engineers love to talk about architecture in qualitative terms.

- *“This module feels tightly coupled”*
- *“That dependency seems risky”*
- *“This design is flexible”*

But beneath those instincts lies something far more concrete
* The **structure of software systems can be measured**
* Architectural problems can be predicted **long before production failures reveal them**.

* The formulas behind these metrics have been around since the 1990s
* They require no machine learning
* Just counting (...and occasionally a little division)


## 🗿The Architecture That Was “Good Enough”… Until It Opened a Hellmouth 👹

"For the first six months, even a year, everything felt fast.  Our Ruby-on-Rails application was humming along."
* Features shipped quickly.
* Bug fixes took hours, not days.
* Engineers felt productive.

>Then, something strange started happening; it began to shift:
* A simple change began taking longer.
* A feature that should have touched one component suddenly required edits across:
  - seemingly-unrelated models and controllers
  - background jobs
  - serializers
  - service classes
  - many tests scattered across the codebase, some that seem unrelated.

_Then the real symptoms appeared:_
* New engineers joined the team and couldn't make heads or tails of the system.
* Bug fixes triggered unrelated failures.
* A “small refactor” broke three features nobody expected to be connected.
* Every change started to feel dangerous.

---

Eventually they bring in an architect, who spends a little while looking at our software and running various tools.  He tells us:

> Your problem isn’t Rails. Your problem is **coupling**.
> All the software enginners had heard of this

Some people remember this from _Software Engineering Principles_ (or whatever the course/book/article you read about this).  It maybe have come up a few times since, but never _exactly quantified_.  Now here it is in the real world.

The application had quietly evolved into something **infamous**:

### A Tightly Coupled Monolith

>  It's a simple complex system.  Because it's simple, it's prone to cascades, and because it's complex, you can't predict what's going to fail. Or how. -- _"The Expanse"_

* A change almost anywhere could trigger side-effects somewhere else.
* Features that should have touched one module required edits across five.
* Bug fixes became archaeology.
### And the surprising part?
* These structural problems weren't mysterious.
* They were **measurable**.
___

## 📐 The Two Numbers That Explain Most Architecture

Nearly every structural coupling metric derives from two simple counts.

```text
        Who depends on me?
                ↑
               Ca
                │
                ● GIVEN MODULE/PACKAGE
                │
               Ce
                ↓
        Upon whom do I depend?
```

### Afferent Coupling (Ca)

```text
Ca = number of external modules that depend on this one
```

Afferent coupling measures **responsibility**.

If many modules depend on you, your stability matters.

Break this module, and others break too.

Modules with high Ca become **structural anchors**.

### Efferent Coupling (Ce)

```text
Ce = number of external modules this module depends on
```

Efferent coupling measures **vulnerability**.

The more dependencies you have, the more ways your code can break.

Every dependency introduces:

- version risk
- semantic assumptions
- upgrade friction

Dependencies are powerful.

But they are never free.

### 🧮 A Simple Analogy

These metrics behave like a financial balance sheet.

| Metric                     | Analogy                        |
|----------------------------|--------------------------------|
| `Ca` *(Afferent Coupling)* | Creditors (who depends on you) |
| `Ce` *(Efferent Coupling)* | Debts (who you depend on)      |

* Modules with many creditors must be **_stable_**.
* Modules with many debts are inherently **_fragile_**.
---

## 🧭 The Instability Index (I)
<img src="../assets/images/metrics-instability-fig1-overview.png"/>
From Ca and Ce we derive a powerful ratio.

```text
I = Ce / (Ce + Ca)
```

### Ranges
Instability ranges from `0` to `1`

| I Range         | Stability  | Meaning                           | Change Strategy  |
|-----------------|------------|-----------------------------------|------------------|
| 0.0 ≤ I < 0.25  | Stable     | Many dependents, few dependencies | Change with care |
| 0.25 ≤ I < 0.50 | Balanced   | Healthy structural position       | Normal dev pace  |
| 0.50 ≤ I < 0.75 | Borderline | Dependency heavy                  | Monitor closely  |
| 0.75 ≤ I ≤ 1.0  | Unstable   | Few dependents, many dependencies | Refactor freely  |

## Instability "Curves"
<img src="../assets/images/metrics-instability-fig2-curves.png">

Modules with **low instability** are depended on by many others.

Modules with **high instability** depend on many others but are depended upon by few.

This leads to one of the most important architectural principles.

### Stable Dependencies Principle

Dependencies should flow **toward stability**.

```text
unstable modules  →  stable modules
```

When stable modules depend on unstable ones, architectural fragility appears quickly.


## 🧬 Abstractness (A)

This metric differentiates types as **concrete** or **abstract** (`interface`/`protocol`/`port`).


```text
A = Na / Nc
```

### Where

* ``Na`` = number of abstract types
* ``Nc`` =`total number of types


### Interpretation
* ``A = 1`` → completely abstract
* ``A = 0`` → completely concrete

### Conclusion
* Abstraction provides flexibility
* Concrete code provides behavior
* Good architecture balances both
---

In [13]:

# ============================================================
# 📊 MAIN SEQUENCE — CONCEPTUAL VIEW
# ============================================================
import plotly.graph_objects as go

title:str = "Main Sequence Concepts"
subtitle:str= "A + I = 1"
distance_from_main:str = "D = Distance from Main Sequence"
fig_main_seq = go.Figure()

# ZONES
zone_font = dict(family="serif", size=13, color="#8B0000")

# Zone of pain
fig_main_seq.add_trace(go.Scatter(
    x=[0, 0, 0.5, 0], y=[0, 0.5, 0, 0],
    fill="toself",
    fillcolor=PALETTE["zone_pain"],
    line=dict(width=0, color="rgba(255, 255, 255, 0.0)"),
    name="Zone of Pain",
    hoverinfo="name",
)).add_annotation(
    x=0.22, y=0.06,
    text="<b>ZONE OF PAIN</b><br>(Stable + Concrete)",
    showarrow=False, font=zone_font, align="center"
)


# Zone of uselessness
fig_main_seq.add_trace(go.Scatter(
    x=[1, 1, 0.5, 1], y=[1, 0.5, 1, 1],
    fill="toself",
    fillcolor=PALETTE["zone_useless"],
    line=dict(width=0, color="rgba(255, 255, 255, 0.0)"),
    name="Zone of Uselessness",
    hoverinfo="name",
)).add_annotation(
    x=0.76, y=0.98,
    text="<b>ZONE OF USELESSNESS</b><br>(Unstable + Abstract)",
    showarrow=False, font=zone_font, align="center", yanchor="top"
)

# Main Sequence Line
fig_main_seq.add_trace(go.Scatter(
    x=[0, 1], y=[1, 0],
    mode="lines",
    line=dict(color=PALETTE["main_seq"], width=4),
    name="Main Sequence",
    text=subtitle
))

# Distance from main sequence (Sample at 0.6, 0.6)
fig_main_seq.add_traces([
  go.Scatter(
    x=[0.5, 0.6],
    y=[0.5, 0.6],
    mode="lines",
    line=dict(color=PALETTE["main_seq"], width=1.2, dash="dot"),
    showlegend=False,
    name="Main Sequence",
    hoverinfo="skip",
  ),
  go.Scatter(x=[0.6], y=[0.6], hoverinfo="skip", name=distance_from_main, marker=dict(color=PALETTE["main_seq"], size=12))
]).add_annotation(x=0.6, y=0.625, text=distance_from_main, font=dict(size=10, family="Monospace"), showarrow=False)



fig_main_seq.update_layout(
    title=dict(
        text="<b>Main Sequence</b>",
        font=dict(size=20, color=PALETTE["axis"], family="cursive"),
        x=0.5,
    ),
    title_subtitle=dict(
      text=subtitle,
      font=dict(family="Monospace")
    ),
    xaxis=dict(
        title="I (Instability) →",
        range=[0, 1],
        dtick=0.1,
        tick0=0,
        gridcolor=PALETTE["grid"],
        zeroline=False,
        color=PALETTE["axis"],
        showline=True,
        linewidth=1,
        linecolor="rgba(70, 60, 45, 0.30)",
        constrain="domain",
    ),
    yaxis=dict(
        title="A (Abstractness) →",
        range=[0, 1],
        dtick=0.1,
        tick0=0,
        scaleanchor="x",
        scaleratio=1,
        gridcolor=PALETTE["grid"],
        zeroline=False,
        color=PALETTE["axis"],
        showline=True,
        linewidth=1,
        linecolor="rgba(70, 60, 45, 0.30)",
        constrain="domain",
    ),
    paper_bgcolor=PALETTE["paper_bg"],
    plot_bgcolor=PALETTE["plot_bg"],
    width=900,
    height=760,
    showlegend=True,
    legend=dict(
        x=1.01, y=1.0,
        bgcolor="rgba(255,255,255,0)",
        borderwidth=0,
        font=dict(size=11, color=PALETTE["axis"]),
    ),
    margin=dict(l=85, r=180, t=80, b=85),
)

fig_main_seq.show()


## 🧪 The Main Sequence

When plotting **Abstractness (A)** against **Instability (I)**, something interesting appears.

Healthy modules tend to cluster along a line defined by:

```text
A + I = 1
```

This line is called the **Main Sequence**.

The first graph below is the *conceptual* view: just the main sequence and the two danger zones. It teaches the terrain first — the ideal balance line, the **Zone of Pain**, and the **Zone of Uselessness**.

The second graph adds real-ish examples and shows each module’s **distance from the main sequence**:

```text
D = |A + I − 1|
```

That distance tells us how far a module has drifted from the balance between **abstractness** and **instability**. The dotted guide lines point to the nearest point on the main sequence, making the deviation visible rather than merely numerical.


## 🪨 Architectural Danger Zones

### Zone of Pain

```text
low A
low I
```

Meaning:

```text
concrete AND stable
```

These modules are depended on by many other modules but contain little abstraction.

Examples often include:

- database schemas
- configuration systems
- foundational libraries

Changing them causes cascading impact.

Hence the name.

### Zone of Uselessness

```text
high A
high I
```

Meaning:

```text
abstract AND unstable
```

These modules contain abstractions nobody uses.

Example:

```text
12 interfaces
1 implementation
0 dependents
```

Beautiful architecture.

No real purpose.

## 🧪 Distance From the Main Sequence

Once we place real modules on the same chart, the picture gets richer.

The detailed graph below shows:

- the **main sequence**
- the two architectural danger zones
- example modules in and out of those zones
- a dotted guide line from each module to its nearest point on the main sequence
- the **distance value** (`D`) for the more interesting examples

That lets us see not just *where* a module sits, but *how far off-balance* it is.

Some modules live outside the danger zones and are still worth watching. A service layer, API gateway, or shared utility package may not be pathological, but a non-zero distance still suggests the design is drifting away from the ideal balance.


In [14]:
# ============================================================
# 📊 DISTANCE FROM MAIN SEQUENCE — DETAILED VIEW
# ============================================================
import plotly.graph_objects as go

for m in DETAILED_MODULES:
    m["D"] = abs(m["A"] + m["I"] - 1)

def project_to_main_sequence(i_value: float, a_value: float):
    proj_i = (i_value - a_value + 1) / 2
    proj_i = max(0, min(1, proj_i))
    proj_a = 1 - proj_i
    return proj_i, proj_a

fig_distance = go.Figure()

# Traces for zones of pain and usefulness and
fig_distance.add_traces([
  go.Scatter(
    x=[0, 0, 0.5, 0], y=[0, 0.5, 0, 0],
    fill="toself",
    fillcolor=PALETTE["zone_pain"],
    line=dict(width=0, color="rgba(255, 255, 255, 0.0)"),
    name="Zone of Pain",
    hoverinfo="name",
  ),

  go.Scatter(
    x=[1, 1, 0.5, 1], y=[1, 0.5, 1, 1],
    fill="toself",
    fillcolor=PALETTE["zone_useless"],
    line=dict(width=0, color="rgba(255, 255, 255, 0.0)"),
    name="Zone of Uselessness",
    hoverinfo="name",
  ),

  go.Scatter(
    x=[0, 1], y=[1, 0],
    mode="lines",
    line=dict(color=PALETTE["main_seq"], width=4),
    name="Main Sequence",
    hovertemplate=subtitle,
  )
])

for m in DETAILED_MODULES:
    proj_i, proj_a = project_to_main_sequence(m["I"], m["A"])
    fig_distance.add_trace(go.Scatter(
        x=[m["I"], proj_i],
        y=[m["A"], proj_a],
        mode="lines",
        line=dict(color=PALETTE["guide"], width=1.2, dash="dot"),
        showlegend=False,
        hoverinfo="skip",
    ))
    fig_distance.add_trace(go.Scatter(
        x=[proj_i], y=[proj_a],
        mode="markers",
        marker=dict(size=5, color="rgba(90,82,70,0.28)", line=dict(width=0)),
        showlegend=False,
        hoverinfo="skip",
    ))

label_positions = {
    "Database Schema": "middle right",
    "Unused Interfaces": "middle right",
    "Core Domain Interfaces": "top left",
    "Domain Model": "middle right",
    "Service Layer": "middle right",
    "Perfect Balance": "middle right",
    "API Gateway": "top right",
    "Shared Utilities": "bottom right",
    "Integration Adapter": "bottom left",
}

for m in DETAILED_MODULES:
    fig_distance.add_trace(go.Scatter(
        x=[m["I"]], y=[m["A"]],
        mode="markers+text",
        marker=dict(size=16, color=m["color"], line=dict(width=2, color="white")),
        text=[m["name"]],
        textposition=label_positions.get(m["name"], "middle right"),
        textfont=dict(size=11, color=PALETTE["axis"]),
        showlegend=False,
        hovertemplate=(
            f"<b>{m['name']}</b><br>"
            f"I: {m['I']:.2f}<br>"
            f"A: {m['A']:.2f}<br>"
            f"D: {m['D']:.2f}<extra></extra>"
        ),
    ))

show_distance_for = {
    "Database Schema", "Unused Interfaces", "Service Layer",
    "API Gateway", "Shared Utilities", "Integration Adapter"
}

for m in DETAILED_MODULES:
    if m["name"] not in show_distance_for or m["D"] <= 0.001:
        continue
    proj_i, proj_a = project_to_main_sequence(m["I"], m["A"])
    mid_i = (m["I"] + proj_i) / 2
    mid_a = (m["A"] + proj_a) / 2
    fig_distance.add_annotation(
        x=mid_i, y=mid_a,
        text=f"D={m['D']:.2f}",
        showarrow=False,
        font=dict(size=9, color="rgba(55,55,55,0.92)"),
        bgcolor=PALETTE["label_bg"],
        bordercolor="rgba(0,0,0,0.10)",
        borderwidth=0.5,
        borderpad=2,
    )

zone_font = dict(family="serif", size=13, color="#8B0000")

fig_distance.add_annotation(
    x=0.22, y=0.06,
    text="<b>ZONE OF PAIN</b><br>(Stable + Concrete)",
    showarrow=False, font=zone_font, align="center"
)
fig_distance.add_annotation(
    x=0.76, y=0.98,
    text="<b>ZONE OF USELESSNESS</b><br>(Unstable + Abstract)",
    showarrow=False, font=zone_font, align="center", yanchor="top"
)

fig_distance.update_layout(
    title=dict(
        text="<b>Distance from Main Sequence</b>",
        font=dict(family='cursive', size=20, color=PALETTE["axis"]),
        x=0.5,
    ),
    xaxis=dict(
        title="I (Instability) →",
        range=[0, 1],
        dtick=0.1,
        tick0=0,
        gridcolor=PALETTE["grid"],
        zeroline=False,
        color=PALETTE["axis"],
        showline=True,
        linewidth=1,
        linecolor="rgba(70, 60, 45, 0.30)",
        constrain="domain",
    ),
    yaxis=dict(
        title="A (Abstractness) →",
        range=[0, 1],
        dtick=0.1,
        tick0=0,
        scaleanchor="x",
        scaleratio=1,
        gridcolor=PALETTE["grid"],
        zeroline=False,
        color=PALETTE["axis"],
        showline=True,
        linewidth=1,
        linecolor="rgba(70, 60, 45, 0.30)",
        constrain="domain",
    ),
    paper_bgcolor=PALETTE["paper_bg"],
    plot_bgcolor=PALETTE["plot_bg"],
    width=980,
    height=780,
    showlegend=True,
    legend=dict(
        x=1.01, y=1.0,
        bgcolor="rgba(255,255,255,0)",
        borderwidth=0,
        font=dict(size=11, color=PALETTE["axis"]),
    ),
    margin=dict(l=85, r=190, t=80, b=85),
)

fig_distance.show()


## 🗺️ Where These Metrics Apply

These metrics apply to many software systems:

- large monoliths
- modular applications
- plugin architectures
- libraries
- microservices

Microservice systems especially benefit from coupling analysis because dependencies often hide behind **network calls rather than imports**.

A service with high efferent coupling may rely on many downstream services.

Each dependency increases operational risk.

Understanding coupling helps prevent systems from drifting toward the dreaded:

```text
distributed monolith
```

## 🧱 The Takeaway

Architecture is often treated as an art.

But beneath the diagrams lies something more mechanical.

Software systems obey structural forces.

Coupling is one of them.

And like gravity…

you can ignore it.

But you cannot escape it.

## 📚 References

- Martin, R. C. (1994). *OO Design Quality Metrics: An Analysis of Dependencies.*
- Martin, R. C. (2017). *Clean Architecture.*
- Lakos, J. (1996). *Large-Scale C++ Software Design.*
- Ford, N., Parsons, R., & Kua, P. (2017). *Building Evolutionary Architectures.*